# Figure U - Spatial embedding with cluster coloring

Cache-first notebook for electrode coordinates colored by LRG clusters.
Legacy reference: spatial embedding plan (Stage 05U).


In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *


## Parameters
Select a patient with implant coordinates and cached LRG results.


In [ ]:
from pathlib import Path

patient = "Pat_02"
band = "beta"
fc_method = "msc"
phase_a = "rsPre"
phase_b = "rsPost"


## Load metadata + cluster labels
This uses cached LRG results and implant coordinates only.


In [ ]:
import numpy as np

from lrg_eegfc.utils.io.patient import load_patient_metadata
from lrg_eegfc.utils.metrics.reorganization import compute_cluster_labels
from lrg_eegfc.workflow.lrg import load_lrg_result

metadata = load_patient_metadata(patient, Path("data/stereoeeg_patients"))
if metadata is None:
    raise FileNotFoundError("Missing implant/channel metadata for patient.")

result_a = load_lrg_result(patient, phase_a, band, fc_method, Path("data/lrg_cache"))
result_b = load_lrg_result(patient, phase_b, band, fc_method, Path("data/lrg_cache"))
if result_a is None or result_b is None:
    raise FileNotFoundError("Missing LRG cache for one or both phases.")

labels_a = compute_cluster_labels(result_a.linkage_matrix, result_a.optimal_threshold)
labels_b = compute_cluster_labels(result_b.linkage_matrix, result_b.optimal_threshold)

metadata = metadata.copy()
metadata["cluster_a"] = labels_a
metadata["cluster_b"] = labels_b


## Plot embedding (matplotlib fallback)
If MNE or Nilearn are installed, you can swap the plotting cell below.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

def _plot_clusters(ax, coords, labels, title):
    unique = np.unique(labels)
    cmap = cm.get_cmap("tab20")
    for idx, lab in enumerate(unique):
        mask = labels == lab
        color = cmap(idx % 20)
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            coords[mask, 2],
            s=40,
            color=color,
            label=f"Cluster {lab}",
            alpha=0.85,
        )
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

coords = metadata[["x", "y", "z"]].to_numpy(dtype=float)
if np.isnan(coords).any():
    raise ValueError("Missing coordinate values in implant metadata.")

fig = plt.figure(figsize=(12, 6))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")

_plot_clusters(ax1, coords, metadata["cluster_a"].to_numpy(), f"{phase_a} clusters")
_plot_clusters(ax2, coords, metadata["cluster_b"].to_numpy(), f"{phase_b} clusters")

fig.suptitle(f"{patient} {band} ({fc_method})", fontsize=14)
fig.tight_layout()

output_dir = Path("data/figures/spatial") / patient
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{band}_{phase_a}_vs_{phase_b}_{fc_method}_clusters.png"
fig.savefig(output_path, dpi=150, bbox_inches="tight")
plt.close(fig)
output_path
